# 05: Offline Recommender Evaluation, Ablation Study & Hybrid Demo
This notebook demonstrates full temporal offline evaluation, ablation study, and the unified RecommendationEngine with cold-start, sparse, and mature customer routing.

In [ ]:
import sys, os
from pathlib import Path
import pandas as pd
import numpy as np

# Robust root resolver whether run from notebooks/ or workspace root
_curr = Path('.').resolve()
PROJECT_ROOT = _curr if (_curr / 'src').exists() else (_curr.parent if (_curr.parent / 'src').exists() else (_curr.parents[1] if len(_curr.parents) > 1 and (_curr.parents[1] / 'src').exists() else _curr))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Switch working directory to project root so all relative paths ('data/...', 'artifacts/...') resolve seamlessly
os.chdir(PROJECT_ROOT)

from src.recommenders.hybrid import RecommendationEngine


## 1. Load Production Recommendation Engine & Evaluation Results

In [ ]:
engine = RecommendationEngine.load('artifacts')
eval_df = pd.read_csv('outputs/metrics/evaluation_metrics.csv')
ablation_df = pd.read_csv('outputs/metrics/ablation_study.csv')
print("Offline Evaluation Summary:")
eval_df


## 2. Component Ablation Study Analysis

In [ ]:
print("Ablation Study Results:")
ablation_df[['model', 'recall@10', 'recall@20', 'hit_rate@10', 'hit_rate@20', 'ndcg@10', 'catalog_coverage']]


## 3. Live Recommendation Engine Demo across Customer States

In [ ]:
# Case A: Brand New Customer (Cold-Start)
cold_res = engine.recommend(customer_id=9999999, n=5)
print(f"Cold-Start User Strategy: {cold_res['recommendation_type']}")
for r in cold_res['recommendations']:
    print(f"  Rank #{r['rank']} Product #{r['product_id']} (Score: {r['score']:.3f}, Sources: {r['sources']}): {r['explanation']}")

# Case B: Mature Customer (Full Hybrid)
mature_user = list(engine.user_interaction_counts.keys())[0]
mature_res = engine.recommend(customer_id=mature_user, n=5)
print(f"\nMature User Strategy ({mature_res['segment_label']}): {mature_res['recommendation_type']}")
for r in mature_res['recommendations']:
    print(f"  Rank #{r['rank']} Product #{r['product_id']} (Score: {r['score']:.3f}, Sources: {r['sources']}): {r['explanation']}")
